<a href="https://colab.research.google.com/github/kajallsingh/Bag_Detection/blob/main/Bag_Counting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install required libraries
!pip install ultralytics opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.2 MB/s eta 0:00:00


In [2]:
# Upload your video
from google.colab import files
uploaded = files.upload()

Saving Problem Statement Scenario2.mp4 to Problem Statement Scenario2.mp4


In [4]:
import cv2
import os

In [11]:
model = YOLO('yolov8n.pt')

In [12]:
# Open video
cap = cv2.VideoCapture(video_path)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

In [13]:
# Output video writer
output_path = "final_output.mp4"
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

In [14]:
# Counting parameters
line_x = width // 2  # Vertical counting line
counted_ids = set()
bag_count = 0
previous_positions = {}

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Track objects
    results = model.track(frame, persist=True)

    if results[0].boxes.id is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        ids = results[0].boxes.id.cpu().numpy()
        classes = results[0].boxes.cls.cpu().numpy()

        for box, track_id, cls in zip(boxes, ids, classes):
            # Only count cement bag class
            if int(cls) == 0:  # 0 = cement_bag in custom dataset
                x1, y1, x2, y2 = map(int, box)
                center_x = int((x1 + x2) / 2)
                center_y = int((y1 + y2) / 2)

                # Draw bounding box
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(frame, f"ID:{int(track_id)}", (x1, y1-10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

                # Initialize previous position
                if track_id not in previous_positions:
                    previous_positions[track_id] = center_x

                prev_x = previous_positions[track_id]

                # RIGHT → LEFT movement (adjust as per your video)
                if prev_x > line_x and center_x <= line_x and track_id not in counted_ids:
                    counted_ids.add(track_id)
                    bag_count += 1

                # Update previous position
                previous_positions[track_id] = center_x

    # Draw counting line
    cv2.line(frame, (line_x, 0), (line_x, height), (0, 0, 255), 3)

    # Show count on video
    cv2.putText(frame, f"Bag Count: {bag_count}", (30, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 3)

    out.write(frame)

cap.release()
out.release()

print(f"✅ Final Bag Count: {bag_count}")
print("Output video saved as final_output.mp4")


0: 640x384 5 persons, 342.1ms
Speed: 15.8ms preprocess, 342.1ms inference, 46.4ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 5 persons, 127.8ms
Speed: 3.1ms preprocess, 127.8ms inference, 1.1ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 5 persons, 123.3ms
Speed: 3.1ms preprocess, 123.3ms inference, 1.1ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 5 persons, 125.0ms
Speed: 3.0ms preprocess, 125.0ms inference, 1.0ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 4 persons, 128.5ms
Speed: 3.2ms preprocess, 128.5ms inference, 1.1ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 5 persons, 121.9ms
Speed: 3.1ms preprocess, 121.9ms inference, 1.0ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 5 persons, 140.4ms
Speed: 3.1ms preprocess, 140.4ms inference, 1.1ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 5 persons, 124.3ms
Speed: 4.2ms preprocess, 124.3ms inference, 1.1ms postprocess pe

In [15]:
# Download output video
files.download("final_output.mp4")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>